# JavaScript Strings and Regular Expressions

A practical guide to defining and creating strings, working with the String API, and applying regular expressions in JavaScript (ECMAScript 2022). The course builds three mini-apps: a currency converter, a string utility library, and a basic spell-checker.

## Table of Contents

- [Module 01: Diving into String Definition](#module-01-diving-into-string-definition)
- [Module 02: Achieving Tasks with the String API](#module-02-achieving-tasks-with-the-string-api)
- [Module 03: Understanding and Applying Regex](#module-03-understanding-and-applying-regex)

In [ ]:
from pathlib import Path
from IPython.display import HTML, display
import base64

def render_diagram(image_path, width=900):
    data = base64.b64encode(Path(image_path).read_bytes()).decode()
    display(HTML(f'<div style="text-align:center"><img src="data:image/png;base64,{data}" width="{width}"/></div>'))

---
## Module 01: Diving into String Definition

_Covers the fundamentals of string creation in JavaScript: quote styles, template literals, special characters, Unicode, and the `String` constructor. Introduces a currency converter app._

### Version and New Features

This course targets **ECMAScript 2022**. Relevant new features covered:

- `BigInt`
- `.at()` method
- `d` regex flag

### JavaScript Engines

JavaScript code is executed by a JS engine. Two widely used engines:

In [ ]:
render_diagram(Path("assets/01-js-engines-v8-spidermonkey.png"))

- **V8** — used by Chrome and Node.js
- **SpiderMonkey** — used by Firefox

### Defining Strings

JavaScript offers three ways to define a string literal:

| Syntax | Example |
|---|---|
| Double quotes | `"hello"` |
| Single quotes | `'hello'` |
| Template literal | `` `hello` `` |

Double and single quotes are functionally identical. **Pick one style and use it consistently across a project.**

In [ ]:
render_diagram(Path("assets/02-consistent-quote-style.png"))

Linting tools that enforce quote consistency:

- **ESLint** — configurable rule `quotes`
- **SonarSource** — enforces consistency as a code-smell rule

### Template Literals

Template literals (backtick strings) support **string interpolation** — embedding expressions directly inside the string:

In [ ]:
render_diagram(Path("assets/03-string-interpolation-arrow.png"))

```javascript
// Old-style concatenation
"The rate is " + 0.9;

// Template literal interpolation — preferred
const rate = 0.9;
`The rate is ${rate}`;
```

**Best practices for template literals:**

- Use template literals *only* when you actually interpolate — don't use backticks for static strings
- Don't nest template literals deeply

```javascript
// avoid: backtick string with no interpolation
"The rate is ${rate}";  // ${rate} printed literally — bug!

// avoid: deep nesting
`We ... ${expression ? `${a}` : b}... text`;
```

### String.raw Tag

`String.raw` is a template tag that prevents escape sequences from being processed. Useful for Windows paths and documentation:

```javascript
const rate = 0.9;
String.raw`Our rate is \t \n ${rate}`;
// Output: Our rate is \t \n 0.9
// \t and \n are NOT converted to tab/newline
```

**Use cases:**
- Tutorials and documentation (showing escape sequences as text)
- Windows directory paths: `C:\Users\`

### Special Characters and Unicode

JavaScript strings are UTF-16 encoded. Every character has a **code point** and a **UTF-16 representation**:

In [ ]:
render_diagram(Path("assets/04-unicode-16bit-units.png"))

| Character | Code Point | UTF-16 | 16-bit units |
|---|---|---|---|
| `A` | 65 | U+0041 | 1 unit |
| `z` | 122 | U+007A | 1 unit |
| 💸 (emoji) | 128184 | 0xD83D, 0xDCB8 | 2 units (surrogate pair) |

Emojis use **surrogate pairs** — two 16-bit units. This means `string.length` counts code units, not visible characters.

**Emoji handling:**
- Support varies by platform — test on all target environments
- Use `.replace(regex)` or a dedicated library to remove emojis
- Avoid reinventing emoji detection — use a battle-tested library

### Summary

- Single or double quotes — pick one, be consistent
- Use template literals for interpolation
- Be careful with special characters and escape sequences
- Use `String.raw\`...\`` to suppress escape processing
- Emojis are surrogate pairs — use a library and test thoroughly
- Avoid `new String("s")` — creates a String object, not a primitive

### Demo: Currency Converter

Demonstrates string definition fundamentals including quote styles, template literals, special characters, Unicode, and the `String` constructor. The starter and complete versions are both included.

See [`01-strings-regex-demos/`](01-strings-regex-demos/) for the full project.

```javascript
// From module2/_1defining-strings.js
let s  = "Currency rate";
let s2 = 'Currency rate';

let mix  = "The rate for 'usd/eur' is 0.9";
let mix2 = 'The rate for "usd/eur" is 0.9';

// Escaping quotes inside strings
console.log("What do you mean by \"code\"");
console.log('It\'s not what I meant');
```

---
## Module 02: Achieving Tasks with the String API

_Covers string length and emptiness checks, comparison operators, type coercion, concatenation, sorting, substringing, and the `.at()` method. Builds a library of string utility functions._

### String Utility Library Overview

The module builds a utility library with these functions:

```javascript
function isEmpty(str)              { /* ... */ }
function isBlank(str)              { /* ... */ }
function sortNumberedStrings(str)  { /* ... */ }
function truncate(str)             { /* ... */ }
```

### Checking Empty and Blank Strings

```javascript
function isEmpty(str) {
    return (!str || str.length === 0);
}
// isEmpty(null)  → true
// isEmpty(false) → true
// isEmpty("")    → true
// isEmpty("hi")  → false
```

Note: `!str` is truthy for `null`, `undefined`, `false`, `0`, and `""` — the `|| str.length === 0` clause handles the empty string case explicitly for clarity.

### Comparing Strings

#### Strict vs. Loose Equality

| Strict (`===`) | Result | Loose (`==`) | Result |
|---|---|---|---|
| `"a" === "a"` | `true` | `"a" == "a"` | `true` |
| `2 === 2` | `true` | `2 == 2` | `true` |
| `"a" === "b"` | `false` | `"a" == "b"` | `false` |
| `2 === 3` | `false` | `2 == 3` | `false` |

**Always prefer `===`** — it never performs type coercion.

#### When Loose Equality Surprises You

```javascript
"0" === 0   // false — different types
"" === 0    // false — different types

"0" == 0    // true  — "0" converted to number 0
"" == 0     // true  — both are "falsy"
```

#### The One Valid Use Case for `==`

```javascript
// Checks for both null and undefined in one expression
if (variable == null) { /* ... */ }

// Equivalent to:
if (variable === undefined || variable === null) { /* ... */ }
```

#### Case-Insensitive and Accent-Aware Comparison

For locale-aware comparison (ignoring case and accents), use `localeCompare`:

```javascript
"a".localeCompare("A", undefined, { sensitivity: "base" }) // 0 → equal
```

### Unicode Case Folding

Some characters have surprising uppercase forms. Example — German sharp S (ß):

- Small: `ß`
- Traditional capital: `SS`
- Modern capital (added later): `ẞ`

This illustrates why locale-aware string operations matter — `"ß".toUpperCase()` returns `"SS"`, not `"ẞ"` in most runtimes.

### Sorting Strings with Embedded Numbers

When sorting an array of numbers, the type matters:

In [ ]:
render_diagram(Path("assets/05-nums-vs-string-nums.png"))

```javascript
let nums  = [1, 2, 3];           // numeric sort → 1, 2, 3
let nums2 = ["1", "2", "3"];    // lexicographic sort → "1", "2", "3"
```

Lexicographic (string) sort compares character codes — `"49"` sorts before `"111"` because `'4' > '1'`:

In [ ]:
render_diagram(Path("assets/06-numbers-as-strings-sort.png"))

For ranked strings like `["1 - Jane", "2 - Joe"]`, use `.localeCompare()` or parse the numeric prefix:

```javascript
let rankings = ["1 - Jane", "2 - Joe"];
rankings.sort((a, b) => a.localeCompare(b, undefined, { numeric: true }));
```

### Concatenation and Type Conversion

The `+` operator is overloaded — it both adds numbers and concatenates strings:

```javascript
console.log("11" + 1);  // "111"  — string + number → string concatenation
console.log("11" - 1);  // 10     — string - number → numeric subtraction
```

For building strings in a loop, modern JavaScript doesn't need `StringBuilder` — string concatenation is optimised by the engine:

```java
// Java — explicit StringBuilder needed for performance
var sb = new StringBuilder();
for (String line : lines) { sb.append(" ").append(line); }
return sb.toString();
```

```javascript
// JavaScript — plain concatenation or Array.join() works fine
const result = lines.join(" ");
```

### Substringing

Three methods exist for extracting substrings:

In [ ]:
render_diagram(Path("assets/07-substringing-methods.png"))

| Method | Signature | Notes |
|---|---|---|
| `substr(start, length)` | `(start, length)` | **Deprecated** — avoid |
| `substring(start, end)` | `(start, end)` | Swaps args if `start > end` |
| `slice(start, end)` | `(start, end)` | Supports negative indices — **preferred** |

⚠️ `substring` silently swaps `start` and `end` if `start > end` — this is a footgun. Use `slice` instead.

In [ ]:
render_diagram(Path("assets/08-slice-highlights.png"))

```javascript
"JavaScript".slice(2);      // "vaScript"  — from index 2 to end
"JavaScript".slice(-2);     // "pt"        — last 2 characters
"JavaScript".slice(2, -2);  // "vaScrip"   — from index 2, excluding last 2
```

### The `.at()` Method (ES2022)

`.at()` provides index access to strings and arrays, supporting **negative indices** (unlike bracket notation):

In [ ]:
render_diagram(Path("assets/09-slice-vs-at.png"))

```javascript
// Arrays
["a", "b", "c"].at(0);   // "a"
["a", "b", "c"].at(2);   // "c"
["a", "b", "c"].at(-1);  // "c"  — last element

// Strings
"abc".at(0);   // "a"
"abc".at(-1);  // "c"  — last character
"abc"[-1];     // undefined  — bracket notation does not support negative indices

// Comparison: .slice(2) returns a substring; .at(2) returns a single character
"JavaScript".slice(2);  // "vaScript"
"JavaScript".at(2);     // "v"
```

### Internationalisation: Number and Currency Formatting

Different locales format numbers differently:

```javascript
// US:  $123,456.00
// GER: 123.456,00 €

new Intl.NumberFormat('en-US', { style: 'currency', currency: 'USD' }).format(123456);
// → "$123,456.00"
```

### Summary

Common string operations covered in this module:

- `str.length` — number of UTF-16 code units
- `===` for strict equality; `==` only for null-check idiom
- `localeCompare()` for locale-aware sorting and comparison
- `slice(start, end)` — preferred substringing method
- `.at(index)` — index access with negative index support (ES2022)

### Demo: String Utility Library

Implements `isEmpty`, `isBlank`, `truncate`, and `sortNumberedStrings`. The complete version adds immutability examples, string comparison, concatenation, and substringing exercises.

See [`01-strings-regex-demos/`](01-strings-regex-demos/) (`module3/` in both starter and complete) for the full project.

```javascript
// From module3/_5substringing.js
"JavaScript".slice(2);      // "vaScript"
"JavaScript".slice(-2);     // "pt"
"JavaScript".slice(2, -2);  // "vaScrip"
```

---
## Module 03: Understanding and Applying Regex

_Covers regular expression creation, flags, the `match` and `replaceAll` methods, the `d` flag for substring indices, and practical tips. Builds a basic spell-checker app._

### What Are Regular Expressions For?

Regular expressions (regex) are patterns used to match, search, and replace text.

**Common use cases:**
- Search and replace
- Match and validate (emails, phone numbers)
- Password strength checking
- Typo and formatting detection

### Creating a Regex

Two syntaxes:

```javascript
// Literal — preferred for known patterns
const re = /pattern/flags;

// Constructor — use when the pattern is dynamic
const re = new RegExp('pattern', 'flags');
```

### Regex Flags

| Flag | Meaning |
|---|---|
| `g` | Global — find all matches, not just the first |
| `i` | Case-insensitive |
| `m` | Multiline — `^` and `$` match line boundaries |
| `u` | Unicode — enables full Unicode matching |
| `y` | Sticky — match must start at `lastIndex` |
| `d` | Indices — include start/end indices for each match (ES2022) |

### Key Regex Methods

```javascript
// .match() with /g flag — returns all matches as an array
"abc abc abc".match(/abc/g);  // ["abc", "abc", "abc"]

// .replaceAll() with regex — replaces all matches
"abc abc abc".replaceAll(/abc/g, "xyz");  // "xyz xyz xyz"
```

The `/g` flag is required with `.replaceAll()` when using a regex — without it, a `TypeError` is thrown.

### Regex Tips

- **Reuse existing solutions** — regex101.com, MDN, and community snippets are invaluable
- **Keep expressions simple** — readable regex is maintainable regex
- **Test extensively** — edge cases in text patterns are easy to miss
- **If too complex, find a non-regex solution** — a well-named function is often clearer

### App: Star Rating and Spell-Checker

The demo builds an interactive star-rating UI and a spell-checker that detects excessive spaces and punctuation errors:

In [ ]:
render_diagram(Path("assets/10-star-rating-app.png"))

### Demo: Spell-Checker

Detects typos using regex — excess spacing between words and spaces before punctuation.

See [`01-strings-regex-demos/`](01-strings-regex-demos/) (`module4/` in both starter and complete) for the full project.

```javascript
// From module4/_5regex_app.js
const inputString = 'It is a good idea to  keep  your    Regex short and simple , right ?';

function checkForTypos(str) {
    let excessiveSpaces = str.match(/\s\s+/g);
    if (excessiveSpaces !== null) {
        console.log(`Excessive spacing found: ${excessiveSpaces.length}`);
    }
    let spaceBeforePunctuation = str.match(/\s(\?|,|\.|!)/g);
    if (spaceBeforePunctuation !== null) {
        console.log(`Space before punctuation: ${spaceBeforePunctuation.length}`);
    }
}

function trimExtraSpace(str) {
    let trimmed = str.replaceAll(/\s\s+/g, ' ');
    return trimmed.replaceAll(/(\s)(\?|,|\.|!)/g, '$2');
}
```

Also demonstrates `replaceAll` with a named pattern:

```javascript
// Replace all occurrences of "Java" with "JavaScript"
"I love Java! Java is great!".replaceAll("Java", "JavaScript");

// Replace numbered list prefixes (e.g. "1-Java", "2-Python") with a regex
"My languages: 1-Java, 2-Python, 3-C#".replaceAll(/\d-\S+/g, "JavaScript");
```